# Small-book quote-or-refuse — finetune (2026-10-13, v4)

Before running: **Runtime -> Change runtime type -> A100 if offered, else L4** (High-RAM not needed).

One session trains **two candidates** (same Qwen2.5-3B base, two refusal
ratios) and downloads **two zips**. When the picker opens, upload **both**
train files at once:
`quote-refuse-2026-10-13.train-x2.jsonl` and `quote-refuse-2026-10-13.train-x1.jsonl`.
Do not upload the bench file.

Watch the two **STOP-CHECK** lines after each training: the reply should be
one quote + one gloss line, then silence. A LONG OUTPUT verdict means tell
the agent before trusting that zip.

If run B crashes at load: Runtime -> Restart session, run cells 1-3 again
(upload both files again), then run the last cell only.

If a cell errors, copy the red text back to the agent; library drift happens.

v4: x3 refusals taught refusals but made some answers loop; this session
narrows the refusal ratio (x2 / x1) to find the balance. v2 fixed rambling
with responses-only loss; v3 proved two candidates fit in one session.

Env fix (same night): a fresh Unsloth build defaulted the trainer to
max_length=1024 and silently dropped 152/167 rows (the assistant marker
was truncated away). The notebook now passes
SFTConfig(max_length=max_seq_length); watch the trainer banner —
'Num examples' must match the card count.

In [ ]:
!pip install -q unsloth

In [ ]:
import gc
import os
import torch

RUNS = [
    {"model": "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
     "export": "smallbook_x2",
     "data": "quote-refuse-2026-10-13.train-x2.jsonl"},  # 167 cards
    {"model": "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
     "export": "smallbook_x1",
     "data": "quote-refuse-2026-10-13.train-x1.jsonl"},  # 140 cards
    # If a model tag 404s: send the red text to the agent; do not invent tags.
]

missing = [r["data"] for r in RUNS if not os.path.exists(r["data"])]
if missing:
    from google.colab import files
    files.upload()  # select BOTH train files in one go
for r in RUNS:
    print(r["data"], "present:", os.path.exists(r["data"]))
assert all(os.path.exists(r["data"]) for r in RUNS), "upload both train files"

In [ ]:
def train_one(model_name, export_name, data_file):
    gc.collect()
    torch.cuda.empty_cache()

    from datasets import load_dataset
    raw = load_dataset("json", data_files=data_file, split="train")
    print(len(raw), "cards loaded from", data_file)

    from unsloth import FastLanguageModel
    from unsloth.chat_templates import train_on_responses_only
    from trl import SFTConfig, SFTTrainer
    from transformers import TrainingArguments

    max_seq_length = 4096
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=max_seq_length,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        lora_alpha=32,
        lora_dropout=0.0,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )

    dataset = raw.map(lambda row: {"text": tokenizer.apply_chat_template(
        row["messages"], tokenize=False, add_generation_prompt=False)})

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        args=SFTConfig(
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            warmup_steps=5,
            max_steps=120,
            max_length=max_seq_length,
            learning_rate=2e-4,
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            logging_steps=5,
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="linear",
            seed=3407,
            output_dir="outputs",
            report_to="none",
        ),
    )
    # Train on the model's ANSWERS only — mask the system prompt and the
    # passages. v1 trained on the whole text and never learned to stop.
    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
    )
    trainer.train()

    # Stop check BEFORE exporting: a short sane reply, then silence.
    FastLanguageModel.for_inference(model)
    prompt_part = dataset[0]["text"].split("<|im_start|>assistant\n")[0] + "<|im_start|>assistant\n"
    inputs = tokenizer([prompt_part], return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=250, do_sample=False)
    reply_ids = out[0][inputs["input_ids"].shape[1]:]
    print("STOP-CHECK reply:\n", tokenizer.decode(reply_ids, skip_special_tokens=False)[:500])
    print("STOP-CHECK verdict:",
          "looks sane" if reply_ids.shape[0] < 250 else
          "LONG OUTPUT - tell the agent before trusting this zip")

    model.save_pretrained_gguf(export_name, tokenizer, quantization_method="q4_k_m")
    import glob, shutil
    from google.colab import files as colab_files
    path = sorted(glob.glob(export_name + "_gguf/*.gguf"))[0]
    print("gguf:", path)
    shutil.make_archive(export_name + "_gguf", "zip", root_dir=path.rsplit("/", 1)[0])
    colab_files.download(export_name + "_gguf.zip")
    print("done:", export_name)

In [ ]:
# Run A: refusals x2 — 167 cards (~20 min A100, ~30-45 min L4)
train_one(RUNS[0]["model"], RUNS[0]["export"], RUNS[0]["data"])

In [ ]:
# Run B: refusals x1 — 140 cards. Same session; memory is freed above.
train_one(RUNS[1]["model"], RUNS[1]["export"], RUNS[1]["data"])